## Check the responses on training data and remove cases which always gives correct answer or never gives correct answer

Author: Sahana Kowshik

Date: 05/13/2025

In [1]:
import os
os.environ['HF_HOME'] = '/projectnb/vkolagrp/skowshik/.cache/'
import torch
import torch.nn.functional as F
import argparse
import pandas as pd
import numpy as np
import torch.distributed as dist
import json
import warnings
import random
import time
import string
warnings.filterwarnings("ignore")
import re

from tqdm import tqdm
from datetime import timedelta
from collections import OrderedDict
from transformers import AutoTokenizer, AutoModel
from vllm import LLM, SamplingParams
os.environ['VLLM_SKIP_P2P_CHECK'] = "1"

INFO 05-13 13:56:46 [__init__.py:239] Automatically detected platform cuda.


In [2]:
# model_id = 'Qwen/Qwen2.5-14B-Instruct'
model_id = 'Qwen/Qwen3-14B'
n_devices = 2
max_model_len = 15000

In [3]:
def load_model(model_id, n_devices, max_model_len):
    """Load VLLM model and Huggingface tokenizer."""
    llm = LLM(
        model=model_id,
        tokenizer=model_id,
        tensor_parallel_size=n_devices,
        gpu_memory_utilization=0.9,
        max_model_len=max_model_len,
        enable_lora=False,
        distributed_executor_backend='mp',
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side='left')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    return llm, tokenizer

In [4]:
llm, tokenizer = load_model(model_id, n_devices, max_model_len)

INFO 05-13 13:56:55 [config.py:717] This model supports multiple tasks: {'classify', 'score', 'reward', 'generate', 'embed'}. Defaulting to 'generate'.
INFO 05-13 13:56:55 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-13 13:56:56 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='Qwen/Qwen3-14B', speculative_config=None, tokenizer='Qwen/Qwen3-14B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=15000, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', reasoning_backend=None), observability_config=ObservabilityConfig(show_hidden_metrics=False, otlp_traces_endpoint=None, coll

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  12% Completed | 1/8 [00:00<00:02,  2.59it/s]
Loading safetensors checkpoint shards:  25% Completed | 2/8 [00:00<00:02,  2.45it/s]
Loading safetensors checkpoint shards:  38% Completed | 3/8 [00:00<00:01,  3.25it/s]
Loading safetensors checkpoint shards:  50% Completed | 4/8 [00:01<00:01,  3.24it/s]
Loading safetensors checkpoint shards:  62% Completed | 5/8 [00:01<00:01,  2.95it/s]
Loading safetensors checkpoint shards:  75% Completed | 6/8 [00:02<00:00,  2.80it/s]
Loading safetensors checkpoint shards:  88% Completed | 7/8 [00:02<00:00,  2.71it/s]


(VllmWorker rank=1 pid=2667548) INFO 05-13 13:57:01 [loader.py:458] Loading weights took 2.72 seconds
(VllmWorker rank=1 pid=2667548) INFO 05-13 13:57:01 [gpu_model_runner.py:1347] Model loading took 13.8818 GiB and 3.087515 seconds


Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:02<00:00,  2.66it/s]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:02<00:00,  2.78it/s]
(VllmWorker rank=0 pid=2667547) 


(VllmWorker rank=0 pid=2667547) INFO 05-13 13:57:02 [loader.py:458] Loading weights took 2.94 seconds
(VllmWorker rank=0 pid=2667547) INFO 05-13 13:57:02 [gpu_model_runner.py:1347] Model loading took 13.8818 GiB and 3.405810 seconds
(VllmWorker rank=1 pid=2667548) INFO 05-13 13:57:10 [backends.py:420] Using cache directory: /usr3/graduate/skowshik/.cache/vllm/torch_compile_cache/5bfd78e597/rank_1_0 for vLLM's torch.compile
(VllmWorker rank=0 pid=2667547) INFO 05-13 13:57:10 [backends.py:420] Using cache directory: /usr3/graduate/skowshik/.cache/vllm/torch_compile_cache/5bfd78e597/rank_0_0 for vLLM's torch.compile
(VllmWorker rank=1 pid=2667548) (VllmWorker rank=0 pid=2667547) INFO 05-13 13:57:10 [backends.py:430] Dynamo bytecode transform time: 7.83 s
INFO 05-13 13:57:10 [backends.py:430] Dynamo bytecode transform time: 7.83 s
(VllmWorker rank=1 pid=2667548) INFO 05-13 13:57:15 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 4.664 s
(VllmWorker

In [5]:
def get_vllm_summary(llm, tokenizer, messages, max_new_tokens=3000):
    """This is a function to generate LLAMA summaries using vllm https://github.com/vllm-project/vllm

    Args:
        llm: LLM object
        tokenizer: Huggingface tokenizer
        input_texts (List): A list of input texts / prompts
        system_msg (str): system message for the LLAMA prompt

    Returns:
        List: A list of generated responses
    """
    
    # return "Answer"
    

    prompts = []
    responses = []
    
    for message in messages:
        input_ids = tokenizer.apply_chat_template(
            message,
            add_generation_prompt=True,
            tokenize=False,
            continue_final_message=False,
            enable_thinking=False
            # return_tensors="pt"
        )
        
        prompts.append(input_ids)
    
    # https://github.com/vllm-project/vllm/blob/main/vllm/sampling_params.py#L38-L66
    sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        min_p=0,
        max_tokens=max_new_tokens,
        # frequency_penalty=0.5,
        # stop=stop_tokens
    )
    
    completions = llm.generate(
        prompts=prompts,
        sampling_params=sampling_params,
    )
    
    for i, output in enumerate(completions):
        temp_gen = output.outputs[0].text
        responses.append(temp_gen)
        
    # print('Successfully finished generating', len(prompts), 'samples!')
    
    return responses

In [6]:
def generate_summary(answers_dicts, llm, tokenizer, max_new_tokens=2048):
    """
    Generate summaries for patient data using a language model.
    
    Args:
    - patient_files: List of strings containing JSON-encoded patient data.
    - system_msg: Initial system message for the language model.
    - llm, tokenizer: Language model and tokenizer for generating summaries.

    Returns:
    - List of patient summaries.
    """
    messages = []
    for answers_dict in answers_dicts:
        message = [
            # {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
            {"role": "user", "content": EXTRACT_ANSWER_PROMPT.format(answer=answers_dict['answer'], question=answers_dict['question'], option=answers_dict['option'])}
        ]
        messages.append(message)
        
    extracted_answers = get_vllm_summary(llm, tokenizer, messages, max_new_tokens=max_new_tokens)
    
    return extracted_answers

In [7]:
def extract_answer_letter(text):
    """
    Extracts the single-letter answer from a string in the format 'ANSWER: X'.

    Args:
        text (str): Input string, e.g., 'ANSWER: B'

    Returns:
        str or None: Extracted answer letter, or None if no match is found.
    """
    match = re.search(r'ANSWER:\s*([A-Z])', text)
    if match:
        return match.group(1)
    return None

In [8]:
def extract_naccid(row):
    naccid = row['problem']['ID']
    row['ID'] = naccid
    return row

def load_results(file_path):
    results = []

    with open(file_path,'r') as f:
        for line in f:
            results.append(json.loads(line))

    results_df = pd.DataFrame(results).explode(['generated_text','finish_reason'])
    results_df = results_df.apply(extract_naccid, axis=1)
    results_df['UNQ_ID'] = [i for i in range(len(results_df))]
    
    return results_df

In [9]:
def extract_final_answer(text):
    # match = re.search(r'<answer>\n(Answer: )([a-zA-Z])\n</answer>', text, re.DOTALL)
    # match = re.search(r'<answer>.*\s*(Answer: )([a-zA-Z]).*\s*</answer>', text, re.DOTALL)
    match = re.search(r'</think>.*\s*(Answer: )([a-zA-Z]).*\s*', text, re.DOTALL)
    return match.group(2).strip().upper() if match else 'invalid'

def extract_final_answer_full_text(text):
    match = re.search(r'.*\s*(Answer: )([a-zA-Z]).*\s*', text, re.DOTALL)
    return match.group(2).strip().upper() if match else 'invalid'

def extract_final_answer_with_answer_tag(text):
    match = re.search(r'<answer>.*\s*(Answer: )([a-zA-Z]).*\s*</answer>', text, re.DOTALL)
    return match.group(2).strip().upper() if match else 'invalid'

In [10]:
def get_invalid(row):
    option_keys = re.findall(r'\b([A-Z])\.', row['problem']['options'])
    if row['prediction'] not in option_keys:
        row["validity"] = "invalid"
    else:
        row["validity"] = "valid"
    return row

In [11]:
def extract_answers_regex_train(results_df, name):
    if "3b" in name:
        results_df['prediction'] = results_df['generated_text'].apply(extract_final_answer_with_answer_tag)
    else:
        results_df['prediction'] = results_df['generated_text'].apply(extract_final_answer)
        
    # results_df['prediction'] = results_df['generated_text'].apply(extract_final_answer)
        
    results_df['ground_truth'] = [p['ground_truth'] for p in results_df.problem]
    
    results_df = results_df.apply(get_invalid, axis=1)
    
    invalid_list[name] = results_df[(results_df['validity'] == 'invalid')]
    invalid_before[name] = len(results_df[(results_df['validity'] == 'invalid')])
    # results_df = results_df[results_df["validity"] == "valid"].reset_index(drop=True)
    results_df.index = results_df.groupby('ID', sort=False).ngroup()
    return results_df

In [12]:
def extract_answers_regex_llm_train(results_df, name):
    results_df['prediction'] = results_df['generated_text'].apply(extract_final_answer)
    results_df = results_df.apply(get_invalid, axis=1)
    invalid_df = results_df[(results_df['validity'] == 'invalid')].reset_index(drop=True)
    invalid_df['extraction_type'] = "llm"
    valid_df = results_df[~results_df['UNQ_ID'].isin(invalid_df['UNQ_ID'])].reset_index(drop=True)
    valid_df['extraction_type'] = "regex"
    
    assert len(valid_df) + len(invalid_df) == len(results_df)
    
    invalid_before[name] = len(invalid_df)
    answers = list(invalid_df['generated_text'])
    questions = [problem['question'] for problem in list(invalid_df['problem'])]
    options = [problem['options'] for problem in list(invalid_df['problem'])]
    answer_dicts = [
        {
            'answer': answers[i],
            'option': options[i],
            'question': questions[i]
        } for i in range(len(invalid_df))
    ]
    extracted_answers = generate_summary(answer_dicts, llm, tokenizer, max_new_tokens=100)
    extracted_answer_letters = [extract_answer_letter(answer) for answer in extracted_answers]
    # extracted_answer_letters = [answer if answer != 'D' else 'invalid' for answer in extracted_answer_letters]
    invalid_df['prediction'] = extracted_answer_letters
    llm_answers[name] = extracted_answers
    
    results_df = pd.concat([valid_df, invalid_df], axis=0).sort_values(by='ID').reset_index(drop=True)
    invalid_after[name] = len(results_df[results_df['prediction'] == 'invalid'])
    # results_df = results_df[results_df['prediction'] != 'invalid'].reset_index(drop=True)
    results_df['ground_truth'] = [p['ground_truth'] for p in results_df.problem]
    results_df.index = results_df.groupby('ID', sort=False).ngroup()
    return results_df

In [13]:
def modify(results_df):
    
    results_combined = results_df.copy()[['prediction','ground_truth', 'ID']]
    results_combined['prediction'] = results_combined['prediction']#.replace({'Y': 'A', 'N': 'B'})
    # results_combined['ground_truth'] = results_combined['ground_truth']#.replace({'Yes': 'A', 'No': 'B'})
    results_combined['correctness'] = results_combined['prediction'] == results_combined['ground_truth']
    # p = len(results_combined) // n
    # results_combined['run'] = list(range(1,p+1))*n
    results_combined = results_combined.reset_index(names=['problem']).reset_index(drop=True)
    return results_combined

In [14]:
def pass_at_k(df, n, k=1): 
    """ 
    :param n: total number of samples 
    :param c: number of correct samples 
    :param k: k in pass@$k$ """
    
    cs = df.groupby('problem').sum('correctness')
    vals = []
    for i, row in cs.iterrows():
        c = row['correctness']
        # print(n,c,k)
        if n - c < k: vals.append(1.0)
        else: vals.append(1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1)))
         
    # print(vals)
    return np.mean(vals)

In [15]:
def get_passat1(df):
    return df.groupby('problem').mean('correctness')[['correctness']].mean().iloc[0]

def get_consatk(df, n):
    return (
        df.groupby('problem')['prediction']
        .apply(lambda x: x.mode()[0]) == 
        df[['problem', 'ground_truth']].drop_duplicates('problem', keep='first')['ground_truth'].reset_index(drop=True)
    ).sum() / n

def combine_results(results_dict, n, k):
    final_dict = {'metric': ['pass@1', 'cons@k']}
    for key, v in results_dict.items():
        # final_dict[k] = [get_passat1(v), get_consat5(v)]
        # print(get_passat1(v), get_consat5(v))
        p = len(v) // n
        final_dict[key] = [pass_at_k(v, p, k), get_consatk(v, n)]

    # Create a tidy DataFrame
    df = pd.DataFrame(final_dict)
    
    return df

In [16]:
def get_mean_length(df):
    lengths = []
    for i, row in df.iterrows():
        lengths.append(len(row['generated_text'].split(" ")))
        
    return sum(lengths) / len(lengths)

In [17]:
EXTRACT_ANSWER_PROMPT = """You will be given a response enclosed within <response> and </response> tags. Your task is to extract the final answer that matches one of the options listed between <options> and </options>, based on the question provided between <question> and </question>. Do not interpret or infer beyond the provided text. Return your answer strictly in this format: ANSWER: <option letter>. Do not output anything else.

<response>
{answer}
</response>

<question>
{question}
</question>

<options>
{option}
</options>"""


# options_list = """A: Normal Cognition
# B: Mild Cognitive Impairment
# C: Dementia
# D. Cannot extract answer from the text"""

# options_list = """A. Yes
# B. No"""

In [18]:
train_results_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/train_results/nacc_train/2025-05-12T212758_e50c886b48594eaa/nacc_train_output.jsonl"
invalid_results_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/train_results/train_invalid/2025-05-13T125255_31b98b5f47ab471b/train_invalid_output.jsonl"

In [19]:
models_dict = {
    'train': train_results_path,
    # 'train': invalid_results_path
}

invalid_before = {
    key:0 for key, value in models_dict.items()
}

invalid_after = {
    key:0 for key, value in models_dict.items()
}

llm_answers = {
    key:0 for key, value in models_dict.items()
}

invalid_list = {
    key:0 for key, value in models_dict.items()
}

In [20]:
model_dfs = {name: load_results(path) for name, path in models_dict.items()}

In [21]:
n = len(model_dfs['train'])

In [22]:
# model_dfs = {name: extract_answers_regex_train(df, name) for name, df in model_dfs.items()} # to use only regex to extract answers
model_dfs = {name: extract_answers_regex_llm_train(df, name) for name, df in model_dfs.items()} # to use both llm and regex to extract answers

Processed prompts: 100%|██████████| 19312/19312 [21:24<00:00, 15.03it/s, est. speed input: 9633.18 toks/s, output: 84.38 toks/s]   


In [23]:
assert len(model_dfs['train']) == n

In [24]:
modified_models = {name: modify(df) for name, df in model_dfs.items()}

In [25]:
assert len(modified_models['train']) == n

In [26]:
summary_df = modified_models['train'].groupby('ID').agg(
    group_size=('correctness', 'size'),
    num_correct=('correctness', 'sum')  # True counts as 1
).reset_index()

In [27]:
all_correct = summary_df[summary_df['num_correct'] == 8].reset_index(drop=True)
all_incorrect = summary_df[(summary_df['num_correct'] == 0) & (summary_df['group_size'] == 8)].reset_index(drop=True)
invalid_df = summary_df[summary_df['group_size'] != 8].reset_index(drop=True)

In [28]:
assert len(invalid_df) == 0

In [29]:
remaining = summary_df[
    (~summary_df['ID'].isin(all_correct['ID'])) &
    (~summary_df['ID'].isin(all_incorrect['ID'])) &
    (~summary_df['ID'].isin(invalid_df['ID']))
].reset_index(drop=True)


In [30]:
train_data = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv")

In [31]:
remaining_data = train_data[train_data['ID'].isin(remaining['ID'])].reset_index(drop=True)
# invalid_data = train_data[train_data['ID'].isin(invalid_df['ID'])].reset_index(drop=True)
all_correct_data = train_data[train_data['ID'].isin(all_correct['ID'])].reset_index(drop=True)
all_incorrect_data = train_data[train_data['ID'].isin(all_incorrect['ID'])].reset_index(drop=True)

In [32]:
# print(f"Number of invalid cases: {len(invalid_data)}")
print(f"Number of all correct cases: {len(all_correct_data)}")
print(f"Number of all incorrect cases: {len(all_incorrect_data)}")
print(f"Number of selected cases: {len(remaining_data)}")

Number of all correct cases: 8637
Number of all incorrect cases: 9133
Number of selected cases: 27271


Number of all correct cases: 8641
Number of all incorrect cases: 9132
Number of selected cases: 27268

In [33]:
assert len(remaining_data) + len(all_correct_data) + len(all_incorrect_data) == len(train_data)

In [34]:
remaining_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/select_train_cases/train_selected.csv", index=False)
# invalid_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/select_train_cases/invalid.csv", index=False)
all_correct_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/select_train_cases/all_correct.csv", index=False)
all_incorrect_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/select_train_cases/all_incorrect.csv", index=False)

In [35]:
remaining_data['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    9857
Primary etiology    9441
MCI subtype         5565
Neuropath           1497
Amyloid PET          795
DATSCAN              116
Name: count, dtype: int64

In [36]:
pet_keys_remove = [
    'Abnormally elevated amyloid on PET',
    'Abnormally low amyloid in CSF',
    'FDG-PET pattern of AD',
    'Tau PET evidence for AD',
    'Abnormally elevated CSF Tau or pTau'
]

In [43]:
for key in pet_keys_remove:
    for i, row in remaining_data[remaining_data['Q_TYPE'] == "Amyloid PET"].iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

In [40]:
dat_keys_remove = [
    'Dopamine transporter scan (DATscan) evidence for Lewy body disease'
]

In [41]:
for key in dat_keys_remove:
    for i, row in remaining_data[remaining_data['Q_TYPE'] == "DATSCAN"].iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break